# Task 03 — Baseline Model Training and Evaluation
**Dataset:** NYC Airbnb 2019  
**Input:** `../task_02/outputs/eda_cleaned.csv`  
**Goal:** Train a reproducible baseline regression model for `price`.  
**SEED:** 42

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

# ── Robust project-root finder ───────────────────────────────────────────────
def _find_project_root() -> Path:
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root — expected a parent containing data/raw/')

PROJECT_ROOT = _find_project_root()
INPUT_PATH   = PROJECT_ROOT / 'agents' / 'claude' / 'task_02' / 'outputs' / 'eda_cleaned.csv'
EDA_SUMMARY  = PROJECT_ROOT / 'agents' / 'claude' / 'task_02' / 'outputs' / 'eda_summary.md'
OUTPUT_DIR   = PROJECT_ROOT / 'agents' / 'claude' / 'task_03' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Input        : {INPUT_PATH}')
print(f'Output dir   : {OUTPUT_DIR}')

---
## 1. Model Selection Reasoning

*(Read EDA summary first, then reason before writing any modelling code.)*

In [ ]:
# ── Print EDA summary for reference ─────────────────────────────────────────
print(EDA_SUMMARY.read_text())

### Model Selection: Ridge Regression on log1p(price)

**What the data tells us:**

From the EDA summary, three facts drive the modelling decision:

1. **`price` is right-skewed** — training a linear model on raw price would mean the loss function is dominated by expensive outliers. Using `log1p(price)` as the target normalises this and makes a linear assumption more defensible.
2. **The strongest predictors are categorical** — `room_type` and `neighbourhood_group` explain most of the variance. A linear model can capture these cleanly via one-hot encoding. Tree-based models would capture non-linear interactions better, but that is the Task 04 improvement.
3. **Numeric features have varying scales** — latitude, longitude, availability, and review counts live on completely different scales. Standardisation is essential for any linear model.

**Why Ridge Regression (not plain OLS)?**

- After one-hot encoding ~220 neighbourhood categories (if included) or 5 boroughs + 3 room types, multicollinearity is a risk. Ridge adds L2 regularisation that shrinks correlated coefficients without eliminating them entirely, making it more stable than OLS for this problem.
- Ridge has a single hyperparameter (`alpha`). For a baseline, we use `alpha=1.0` (sklearn default) — no tuning.
- It trains in milliseconds, is fully interpretable, and gives a clean lower-bound benchmark for Task 04 to beat.

**Known limitations:**

- **Cannot capture non-linear interactions** — e.g., the room-type × borough interaction identified in EDA requires a product feature or a tree-based model.
- **High-cardinality neighbourhood column excluded** — including all 200+ neighbourhoods as dummies would bloat the feature space; for this baseline we use `neighbourhood_group` (5 boroughs) only.
- **Linear assumption on log scale** — the relationship between coordinates and log-price is not perfectly linear, so geo signal will be partially captured but not fully.

**Evaluation approach:**

- Train on `log1p(price)`, back-transform predictions with `np.expm1()` for reporting in the original price space.
- Report RMSE and MAE in original USD (interpretable to stakeholders) and R² on the log scale (reflects model fit quality).
- RMSE on raw price would be inflated by the remaining high-price tail — reporting alongside MAE gives a fairer picture.

In [ ]:
# ── Save model selection reasoning to outputs/ ────────────────────────────────
reasoning_text = """# Model Selection Reasoning — Task 03

## Chosen Model
**Ridge Regression** trained on **log1p(price)** as the target.

## Why log1p(price) as the target?
- Raw `price` is strongly right-skewed (skewness > 4). Training on raw price causes the
  squared-error loss to be dominated by expensive outliers, harming generalisation.
- log1p(price) is approximately normal — a much better fit for the linear model assumption.
- Predictions are back-transformed with np.expm1() for reporting in USD.

## Why Ridge and not plain OLS?
- After one-hot encoding categorical features, mild multicollinearity is expected.
- Ridge L2 regularisation stabilises coefficients without discarding features.
- alpha=1.0 (default) — no hyperparameter tuning; this is a baseline, not the final model.

## Why not a tree-based model as the baseline?
- Trees naturally handle non-linearities and interactions — that would skip the baseline step.
- A linear model gives a clear, interpretable lower bound. Task 04 can show the uplift
  from using a more expressive model (e.g. Random Forest, Gradient Boosting).

## Known Limitations
1. Cannot model the room_type × neighbourhood interaction without an explicit feature.
2. `neighbourhood` column excluded (200+ levels) — using borough-level grouping instead.
3. Geographic signal (latitude/longitude) captured linearly — spatial non-linearities missed.
4. Assumes additive feature contributions on the log scale.

## Features Used
- Categorical (one-hot): room_type, neighbourhood_group
- Numeric (standardised): latitude, longitude, minimum_nights, number_of_reviews,
  reviews_per_month, calculated_host_listings_count, availability_365
- Excluded: neighbourhood (high cardinality — Task 04 improvement)
"""

(OUTPUT_DIR / 'model_selection_reasoning.md').write_text(reasoning_text)
print('Saved → model_selection_reasoning.md')

---
## 2. Load Data & Train/Test Split

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── Define features and target ────────────────────────────────────────────────
CAT_FEATURES = ['room_type', 'neighbourhood_group']
NUM_FEATURES = [
    'latitude', 'longitude', 'minimum_nights',
    'number_of_reviews', 'reviews_per_month',
    'calculated_host_listings_count', 'availability_365'
]
TARGET_RAW = 'price'
TARGET_LOG = 'log_price'

df[TARGET_LOG] = np.log1p(df[TARGET_RAW])

X = df[CAT_FEATURES + NUM_FEATURES]
y = df[TARGET_LOG]

print(f'Features : {X.shape[1]} ({len(CAT_FEATURES)} categorical, {len(NUM_FEATURES)} numeric)')
print(f'Samples  : {len(X)}')

### Split Strategy

**80/20 train/test split, `random_state=SEED=42`.**

- **80/20** is standard for tabular regression at this sample size (~47k rows). 80% (~38k) gives the model sufficient data to learn borough- and room-type-level patterns; 20% (~9.4k) is large enough for stable metric estimates.
- **No stratification on price** — price is continuous, so stratification is not applicable. A random split is appropriate.
- **No cross-validation** at this stage — the baseline's purpose is to establish a reproducible point estimate, not to optimise. Task 04 may introduce CV for model selection.
- **Reproducibility** guaranteed by `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print(f'Train : {X_train.shape[0]} rows  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test  : {X_test.shape[0]} rows  ({X_test.shape[0]/len(X)*100:.1f}%)')

---
## 3. Preprocessing Pipeline

### Pipeline Choices

| Step | Feature type | Transformer | Reason |
|---|---|---|---|
| One-hot encoding | Categorical | `OneHotEncoder(handle_unknown='ignore')` | Converts room_type and neighbourhood_group to dummy variables; `handle_unknown='ignore'` prevents errors if test set has unseen levels |
| Standardisation | Numeric | `StandardScaler()` | Ridge regression is sensitive to feature scale — standardising ensures the regularisation penalty is applied equally across all coefficients |

**Fitted on training data only** — `pipeline.fit()` is called on `X_train` exclusively. `X_test` is only passed to `pipeline.transform()` (via `predict()`), so there is no data leakage.

In [ ]:
# ── Build preprocessing + model pipeline ─────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ('num', StandardScaler(), NUM_FEATURES),
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=SEED)),
])

print('Pipeline:')
print(pipeline)

---
## 4. Train Baseline Model

In [ ]:
# ── Fit on training data only ────────────────────────────────────────────────
pipeline.fit(X_train, y_train)
print('Model trained.')

# ── Feature names after preprocessing ────────────────────────────────────────
cat_names = list(pipeline.named_steps['preprocessor']
                 .named_transformers_['cat']
                 .get_feature_names_out(CAT_FEATURES))
feature_names = cat_names + NUM_FEATURES
print(f'Total features after encoding: {len(feature_names)}')

In [ ]:
# ── Inspect coefficients ─────────────────────────────────────────────────────
coefs = pd.Series(
    pipeline.named_steps['model'].coef_,
    index=feature_names
).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
coefs.plot(kind='barh', ax=ax,
           color=['tomato' if v < 0 else 'steelblue' for v in coefs])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Ridge Regression Coefficients (log1p price scale)')
ax.set_xlabel('Coefficient value')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_coefficients.png')
plt.show()
print('Saved → model_coefficients.png')

---
## 5. Evaluation

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_train_pred_log = pipeline.predict(X_train)
y_test_pred_log  = pipeline.predict(X_test)

# Back-transform to original price space
y_train_pred = np.expm1(y_train_pred_log)
y_test_pred  = np.expm1(y_test_pred_log)
y_train_true = np.expm1(y_train.values)
y_test_true  = np.expm1(y_test.values)

def evaluate(y_true, y_pred, y_true_log, y_pred_log, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true_log, y_pred_log)   # R² on log scale — reflects model fit
    r2_raw = r2_score(y_true, y_pred)
    print(f'  {label}')
    print(f'    RMSE (USD)     : ${rmse:.2f}')
    print(f'    MAE  (USD)     : ${mae:.2f}')
    print(f'    R²   (log)     : {r2:.4f}')
    print(f'    R²   (raw USD) : {r2_raw:.4f}')
    return {'split': label, 'RMSE_USD': round(rmse, 2), 'MAE_USD': round(mae, 2),
            'R2_log': round(r2, 4), 'R2_raw': round(r2_raw, 4)}

print('Metrics:')
train_metrics = evaluate(y_train_true, y_train_pred,
                         y_train.values, y_train_pred_log, 'Train')
test_metrics  = evaluate(y_test_true,  y_test_pred,
                         y_test.values, y_test_pred_log,  'Test')

### Metric Choices — Rationale

| Metric | Scale | Why reported |
|---|---|---|
| **RMSE (USD)** | Original price | Industry-standard; penalises large errors — useful for comparing models |
| **MAE (USD)** | Original price | More robust than RMSE to the remaining price tail; easier to interpret ('on average, off by $X') |
| **R² (log scale)** | Log price | Reflects true model fit — raw-price R² is distorted by the skewed distribution |
| **R² (raw USD)** | Original price | Reported for completeness; expected to be lower due to skew |

**Why not MAPE?** With prices near zero, MAPE becomes unstable. We avoided it.

In [ ]:
# ── Predicted vs Actual ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_true, y_pred, label in [
    (axes[0], y_train_true, y_train_pred, 'Train'),
    (axes[1], y_test_true,  y_test_pred,  'Test')
]:
    ax.scatter(y_true, y_pred, alpha=0.2, s=5, color='steelblue')
    lim = [0, max(y_true.max(), y_pred.max()) * 1.05]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f'Predicted vs Actual — {label}')
    ax.set_xlabel('Actual Price (USD)')
    ax.set_ylabel('Predicted Price (USD)')
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'predicted_vs_actual.png')
plt.show()
print('Saved → predicted_vs_actual.png')

In [ ]:
# ── Residual plots ────────────────────────────────────────────────────────────
residuals_train = y_train_true - y_train_pred
residuals_test  = y_test_true  - y_test_pred
residuals_log_test = y_test.values - y_test_pred_log

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs predicted (test, USD)
axes[0].scatter(y_test_pred, residuals_test, alpha=0.2, s=5, color='darkorange')
axes[0].axhline(0, color='red', linewidth=1.2, linestyle='--')
axes[0].set_title('Residuals vs Predicted — Test (USD)')
axes[0].set_xlabel('Predicted Price (USD)')
axes[0].set_ylabel('Residual (USD)')

# Residuals distribution (test, USD)
axes[1].hist(residuals_test, bins=80, color='mediumseagreen', edgecolor='white')
axes[1].axvline(0, color='red', linewidth=1.2, linestyle='--')
axes[1].set_title('Residual Distribution — Test (USD)')
axes[1].set_xlabel('Residual (USD)')
axes[1].set_ylabel('Count')

# Residuals on log scale (test)
axes[2].scatter(y_test_pred_log, residuals_log_test, alpha=0.2, s=5, color='steelblue')
axes[2].axhline(0, color='red', linewidth=1.2, linestyle='--')
axes[2].set_title('Residuals vs Predicted — Test (log scale)')
axes[2].set_xlabel('Predicted log1p(Price)')
axes[2].set_ylabel('Residual (log)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residual_plots.png')
plt.show()
print('Saved → residual_plots.png')

### Diagnostic Interpretation

**Predicted vs Actual:** Points should cluster along the red diagonal. Scatter above the line = underestimates; below = overestimates. Expect the model to struggle with very high-price listings — the linear model cannot fully capture the tail even on the log scale.

**Residuals vs Predicted:** Ideally a horizontal band around zero. A funnel shape (variance increasing with price) would indicate heteroscedasticity — common in price data and a known limitation of linear regression.

**Residual distribution:** Should be approximately normal and centred at zero. A heavy right tail indicates systematic underestimation of high-price listings — expected given that we excluded neighbourhood-level granularity.

---
## 6. Save Everything for Task 04

In [ ]:
# ── Save baseline results CSV ─────────────────────────────────────────────────
results_df = pd.DataFrame([train_metrics, test_metrics])
results_path = OUTPUT_DIR / 'baseline_results.csv'
results_df.to_csv(results_path, index=False)
print(f'Saved → baseline_results.csv')
print(results_df.to_string(index=False))

In [ ]:
# ── Save fitted pipeline as .pkl ──────────────────────────────────────────────
pipeline_path = OUTPUT_DIR / 'baseline_pipeline.pkl'
with open(pipeline_path, 'wb') as f:
    pickle.dump(pipeline, f)
print(f'Saved → baseline_pipeline.pkl')

# ── Save train/test indices for reproducibility in Task 04 ────────────────────
split_meta = {
    'train_indices': list(X_train.index),
    'test_indices' : list(X_test.index),
    'seed'         : SEED,
    'test_size'    : 0.2,
    'cat_features' : CAT_FEATURES,
    'num_features' : NUM_FEATURES,
    'target'       : TARGET_LOG,
}
split_path = OUTPUT_DIR / 'split_meta.pkl'
with open(split_path, 'wb') as f:
    pickle.dump(split_meta, f)
print(f'Saved → split_meta.pkl  (train={len(X_train)}, test={len(X_test)})')

In [ ]:
# ── Final output checklist ────────────────────────────────────────────────────
expected = [
    'model_selection_reasoning.md',
    'model_coefficients.png',
    'predicted_vs_actual.png',
    'residual_plots.png',
    'baseline_results.csv',
    'baseline_pipeline.pkl',
    'split_meta.pkl',
]

print('Output checklist:')
all_ok = True
for f in expected:
    exists = (OUTPUT_DIR / f).exists()
    status = '  PASS' if exists else '  MISSING'
    print(f'  {status}  {f}')
    if not exists:
        all_ok = False

print()
print('All outputs present!' if all_ok else 'WARNING: some outputs missing — re-run cells above.')